In [4]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
df=pd.read_csv('/content/batting_Features.csv')
df.columns

Index(['date', 'match_id', 'bat_team', 'bowl_team', 'inns', 'batter_id',
       'batter', 'runs', 'balls', 'fours', 'sixes', 'sr', 'isout',
       'wickettype', 'bowler_id', 'bowler', 'venue', 'boundaries',
       'batter_avg', 'batter_rpm', 'batter_sr', 'batter_bpm', 'bat_team_avg',
       'batter_avg_opp', 'batter_rpm_opp', 'batter_bpm_opp', 'batter_sr_opp',
       'bat_team_avg_opp', 'all_team_avg_opp', 'batter_avg_inn',
       'batter_rpm_inn', 'batter_bpm_inn', 'batter_sr_inn', 'bat_team_avg_inn',
       'all_team_avg_inn', 'batter_avg_venue', 'batter_sr_venue',
       'batter_rpm_venue', 'batter_bpm_venue', 'bat_team_avg_venue',
       'bat_team_avg_venue.1', 'all_team_avg_venue', 'all_team_avg_venue_opp',
       'batter_rpm_last5', 'batter_bpm_last5', 'batter_avg_last5',
       'batter_sr_last5'],
      dtype='object')

In [6]:
# Define the conditions for each column
conditions = [
    (df['runs'] >= 25) & (df['runs'] < 50),
    (df['runs'] >= 50) & (df['runs'] < 75),
    (df['runs'] >= 75) & (df['runs'] < 100),
    (df['runs'] >= 100) & (df['runs'] < 125),
    (df['runs'] >= 125) & (df['runs'] < 150),
    (df['runs'] >= 150),
    (df['isout'] & (df['runs'] == 0))  # 'isout' instead of 'isOut'
]

# Define the corresponding column names
column_names = ['25s', '50s', '75s', '100s', '125s', '150s', 'Duck']

# Initialize all new columns with 0
for col in column_names:
    df[col] = 0

# Assign 1 to the appropriate column based on conditions
for condition, col in zip(conditions, column_names):
    df.loc[condition, col] = 1


In [9]:
# Calculate batting fantasy points
df['batting_fp'] = (
    df['runs'] * 1 +  # Runs: +1 per run
    df['fours'] * 4 +  # Boundary Bonus: +4 per four
    df['sixes'] * 6 +  # Six Bonus: +6 per six
    df['25s'] * 4 +  # 25s: +4 pts
    df['50s'] * 8 +  # 50s: +8 pts
    df['75s'] * 12 +  # 75s: +12 pts
    df['100s'] * 16 +  # 100s: +16 pts
    df['125s'] * 20 +  # 125s: +20 pts
    df['150s'] * 24 +  # 150s: +24 pts
    df['Duck'] * -3  # Duck: -3 pts
)

# Add strike rate bonus/penalty
def strike_rate_points(row):
    if row['balls'] >= 20:  # Check if at least 20 balls were played
        if row['sr'] > 140:  # 'sr' instead of 'strikeRate'
            return 6
        elif 120.01 <= row['sr'] <= 140:
            return 4
        elif 100 <= row['sr'] <= 120:
            return 2
        elif 40 <= row['sr'] < 50:
            return -2
        elif 30 <= row['sr'] < 40:
            return -4
        elif row['sr'] < 30:
            return -6
    return 0  # Return 0 if fewer than 20 balls were played

# Apply strike rate points
df['strikeRate_points'] = df.apply(strike_rate_points, axis=1)
df['batting_fp'] += df['strikeRate_points']

# Drop the temporary strikeRate_points column if not needed
df.drop(columns=['strikeRate_points'], inplace=True)
df.head()


,date,match_id,bat_team,bowl_team,inns,batter_id,batter,runs,balls,fours,...,batter_avg_last5,batter_sr_last5,25s,50s,75s,100s,125s,150s,Duck,batting_fp
0,2015-04-08,829705,Mumbai Indians,Kolkata Knight Riders,1,5334,Aaron Finch,5,5,1,...,5.0,100.000000,0,0,0,0,0,0,0,9
1,2015-04-08,829705,Mumbai Indians,Kolkata Knight Riders,1,333904,Aditya Tare,7,7,1,...,7.0,100.000000,0,0,0,0,0,0,0,11
2,2015-04-08,829705,Mumbai Indians,Kolkata Knight Riders,1,33141,Ambati Rayudu,0,2,0,...,0.0,0.000000,0,0,0,0,0,0,1,-3
3,2015-04-08,829705,Kolkata Knight Riders,Mumbai Indians,2,276298,Andre Russell,0,0,0,...,41.0,241.176471,0,0,0,0,0,0,0,0
4,2015-04-08,829705,Mumbai Indians,Kolkata Knight Riders,1,277662,Corey Anderson,55,41,4,...,55.0,134.146341,0,1,0,0,0,0,0,101


In [17]:
df[df['batting_fp'].isnull()].info()


<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 42 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   date                    0 non-null      object 
 1   match_id                0 non-null      int64  
 2   bat_team                0 non-null      object 
 3   bowl_team               0 non-null      object 
 4   inns                    0 non-null      int64  
 5   batter_id               0 non-null      int64  
 6   runs                    0 non-null      int64  
 7   balls                   0 non-null      int64  
 8   isout                   0 non-null      int64  
 9   bowler_id               0 non-null      object 
 10  venue                   0 non-null      object 
 11  boundaries              0 non-null      int64  
 12  batter_avg              0 non-null      float64
 13  batter_rpm              0 non-null      float64
 14  batter_sr               0 non-null      float64
 15  batter

In [29]:
batting_fp = df.drop(columns=['inns', 'bat_team',  'isout'])
batting_fp.head()

,date,match_id,bowl_team,batter_id,runs,balls,bowler_id,venue,boundaries,batter_avg,...,batter_bpm_venue,bat_team_avg_venue,bat_team_avg_venue.1,all_team_avg_venue,all_team_avg_venue_opp,batter_rpm_last5,batter_bpm_last5,batter_avg_last5,batter_sr_last5,batting_fp
0,2015-04-08,829705,Kolkata Knight Riders,5334,5,5,Morne Morkel,Eden Gardens,1,5.0,...,NaN,165.00,165.00,310.900000,NaN,5.0,1.0,5.0,100.000000,9
1,2015-04-08,829705,Kolkata Knight Riders,333904,7,7,Shakib Al Hasan,Eden Gardens,1,7.0,...,NaN,165.00,165.00,316.550000,NaN,7.0,1.0,7.0,100.000000,11
2,2015-04-08,829705,Kolkata Knight Riders,33141,0,2,Morne Morkel,Eden Gardens,0,0.0,...,NaN,165.00,165.00,321.000000,NaN,0.0,0.0,0.0,0.000000,-3
3,2015-04-08,829705,Mumbai Indians,276298,0,0,NaN,Eden Gardens,0,107.0,...,NaN,166.00,166.00,331.000000,NaN,0.0,0.0,41.0,241.176471,0
4,2015-04-08,829705,Kolkata Knight Riders,277662,55,41,NaN,Eden Gardens,7,60.0,...,NaN,173.75,173.75,299.277778,NaN,55.0,7.0,55.0,134.146341,101


In [37]:
batting_fp = df[['match_id', 'batter_id','batting_fp']].copy()
batting_fp

,match_id,batter_id,batting_fp
0,829705,5334,9
1,829705,333904,11
2,829705,33141,-3
3,829705,276298,0
4,829705,277662,101
...,...,...,...
14263,1426312,802575,0
14264,1426312,530011,-3
14265,1426312,1209292,0
14266,1426312,1108375,0


In [38]:
batting_fp.to_csv('batting_fp2.csv', index=False)